In [1]:
from dowhy import CausalModel

def evaluate_causal_constraints(df, domain="walmart"):
    if domain == "walmart":
        # DAG: Unemployment (Macro) -> Sales Risk, mediated by Store Size/Type
        model = CausalModel(
            data=df.dropna(subset=['Unemployment', 'CPI', 'low_sales_risk']),
            treatment='Unemployment',
            outcome='low_sales_risk',
            common_causes=['Store', 'Dept', 'IsHoliday', 'CPI'],
            effect_modifiers=['Type', 'Size'] 
        )
    elif domain == "dataco":
        # DAG: GSCPI (Macro) -> Late Delivery, mediated by Shipping Mode
        model = CausalModel(
            data=df.dropna(subset=['GSCPI', 'late_delivery_risk']),
            treatment='GSCPI',
            outcome='late_delivery_risk',
            common_causes=['Shipping Mode', 'Order Region', 'Category Name']
        )
        
    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(
        identified_estimand, 
        method_name="backdoor.linear_regression"
    )
    
    print(f"[{domain.upper()}] Causal Estimate of Treatment on Risk: {estimate.value:.4f}")
    return estimate